In [1]:
# ============================================
# DÉFI QUOTIDIEN : CLASSIFICATION CHATS VS CHIENS
# GUIDE COMPLET - UN SEUL BLOC
# ============================================

# ============================================
# PARTIE 1 : IMPORTATION DES BIBLIOTHÈQUES
# ============================================
import os, math, re, random, json, yaml
from glob import glob
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configuration
np.random.seed(42)
tf.random.set_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("="*50)
print("DÉFI QUOTIDIEN : CHATS VS CHIENS")
print("CLASSIFICATION BINAIRE AVEC CNN")
print("="*50)
print(f"TensorFlow version: {tf.__version__}")
print(f"GPU disponible: {tf.config.list_physical_devices('GPU')}")

# ============================================
# PARTIE 2 : CHARGEMENT DES DONNÉES
# ============================================
print("\n" + "="*50)
print("CHARGEMENT DES DONNÉES")
print("="*50)

# Configuration des chemins
DATA_ROOT = Path("data/cats_dogs")
train_dir = (DATA_ROOT / "train" / "train") if (DATA_ROOT / "train" / "train").exists() else (DATA_ROOT / "train")
test_dir = (DATA_ROOT / "test" / "test") if (DATA_ROOT / "test" / "test").exists() else (DATA_ROOT / "test")

IMG_HEIGHT, IMG_WIDTH = 180, 180
batch_size = 32
seed = 1337

print(f"Dossier d'entraînement: {train_dir}")
print(f"Dossier de test: {test_dir}")

# Fonction pour construire le DataFrame
def build_df_from_folder(folder: Path, labeled: bool=True):
    exts = ('*.jpg','*.jpeg','*.png','*.bmp')
    files = []
    for ex in exts:
        files.extend(glob(str(folder / '**' / ex), recursive=True))
    if not files:
        raise FileNotFoundError(f"Aucune image trouvée dans {folder}")
    rows = []
    for f in files:
        if labeled:
            name = Path(f).name.lower()
            parent = Path(f).parent.name.lower()
            if parent in {"cat","cats"}:
                label = "cat"
            elif parent in {"dog","dogs"}:
                label = "dog"
            else:
                if re.search(r'(^|[^a-z])cat([^a-z]|$)', name):
                    label = "cat"
                elif re.search(r'(^|[^a-z])dog([^a-z]|$)', name):
                    label = "dog"
                else:
                    continue
            rows.append({"filepath": f, "label": label})
        else:
            rows.append({"filepath": f})
    return pd.DataFrame(rows)

# Création des DataFrames
df_train_full = build_df_from_folder(train_dir, labeled=True)
df_test_full = build_df_from_folder(test_dir, labeled=False)
print(f"Images d'entraînement: {len(df_train_full)}")
print(f"Images de test: {len(df_test_full)}")

# Division train/validation
df_tr, df_val = train_test_split(
    df_train_full, test_size=0.2, stratify=df_train_full["label"], random_state=seed
)
print(f"Train: {len(df_tr)}, Validation: {len(df_val)}")

# ============================================
# PARTIE 3 : GÉNÉRATEURS ET AUGMENTATION
# ============================================
print("\n" + "="*50)
print("CRÉATION DES GÉNÉRATEURS")
print("="*50)

# Générateurs
train_gen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=45,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.5,
    horizontal_flip=True,
    fill_mode='nearest'
)
val_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_flow = train_gen.flow_from_dataframe(
    df_tr, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=True, seed=seed, validate_filenames=False
)
val_flow = val_gen.flow_from_dataframe(
    df_val, x_col="filepath", y_col="label",
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode="binary", batch_size=batch_size,
    shuffle=False, validate_filenames=False
)
test_flow = test_gen.flow_from_dataframe(
    df_test_full, x_col="filepath", y_col=None,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    class_mode=None, batch_size=batch_size,
    shuffle=False, validate_filenames=False
)

print(f"Train: {train_flow.samples} images")
print(f"Validation: {val_flow.samples} images")
print(f"Test: {test_flow.samples} images")

# ============================================
# PARTIE 4 : EXPLORATION DES DONNÉES
# ============================================
print("\n" + "="*50)
print("EXPLORATION DES DONNÉES")
print("="*50)

# Distribution des classes
train_counts = df_tr['label'].value_counts()
print("Distribution des classes - Entraînement:")
print(f"Chats: {train_counts.get('cat', 0)} ({train_counts.get('cat', 0)/len(df_tr)*100:.1f}%)")
print(f"Chiens: {train_counts.get('dog', 0)} ({train_counts.get('dog', 0)/len(df_tr)*100:.1f}%)")

# Visualisation des exemples
def visualize_sample_images(flow, num_images=9):
    images, labels = next(flow)
    fig, axes = plt.subplots(3, 3, figsize=(10, 10))
    class_names = {v: k for k, v in flow.class_indices.items()}
    for i, ax in enumerate(axes.flat):
        if i < num_images:
            ax.imshow(images[i])
            label_idx = int(labels[i])
            label = class_names.get(label_idx, 'Inconnu')
            ax.set_title(f'{label}', fontsize=12, color='green' if label == 'cat' else 'blue')
        ax.axis('off')
    plt.suptitle('Exemples d\'images d\'entraînement', fontsize=16)
    plt.tight_layout()
    plt.show()

visualize_sample_images(train_flow)

# ============================================
# PARTIE 5 : ARCHITECTURE DU MODÈLE
# ============================================
print("\n" + "="*50)
print("ARCHITECTURE DU MODÈLE CNN")
print("="*50)

def create_cats_dogs_model(input_shape=(180, 180, 3)):
    model = tf.keras.Sequential([
        # Bloc 1
        layers.Conv2D(32, (3, 3), activation='relu', padding='same', input_shape=input_shape),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Bloc 2
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Bloc 3
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Bloc 4
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Bloc 5
        layers.Conv2D(512, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),
        
        # Partie dense
        layers.GlobalAveragePooling2D(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

# Création et compilation
model = create_cats_dogs_model()
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print("Modèle créé avec succès!")
model.summary()

# ============================================
# PARTIE 6 : CALLBACKS ET ENTRAÎNEMENT
# ============================================
print("\n" + "="*50)
print("ENTRAÎNEMENT DU MODÈLE")
print("="*50)

callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint('best_cats_dogs_model.h5', monitor='val_accuracy', save_best_only=True, verbose=1)
]

EPOCHS = 50
history = model.fit(
    train_flow,
    steps_per_epoch=train_flow.samples // batch_size,
    epochs=EPOCHS,
    validation_data=val_flow,
    validation_steps=val_flow.samples // batch_size,
    callbacks=callbacks_list,
    verbose=1
)

# ============================================
# PARTIE 7 : VISUALISATION DES COURBES
# ============================================
print("\n" + "="*50)
print("VISUALISATION DES PERFORMANCES")
print("="*50)

def plot_training_history(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
    ax1.plot(history.history['accuracy'], label='Entraînement', color='blue')
    ax1.plot(history.history['val_accuracy'], label='Validation', color='red')
    ax1.set_title('Précision du modèle')
    ax1.set_xlabel('Époque')
    ax1.set_ylabel('Précision')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    ax2.plot(history.history['loss'], label='Entraînement', color='blue')
    ax2.plot(history.history['val_loss'], label='Validation', color='red')
    ax2.set_title('Perte du modèle')
    ax2.set_xlabel('Époque')
    ax2.set_ylabel('Perte')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_training_history(history)

# Analyse du surapprentissage
train_acc = history.history['accuracy'][-1]
val_acc = history.history['val_accuracy'][-1]
print(f"Précision finale - Entraînement: {train_acc:.4f}")
print(f"Précision finale - Validation: {val_acc:.4f}")
print(f"Écart: {(train_acc - val_acc)*100:.2f}%")

# ============================================
# PARTIE 8 : ÉVALUATION SUR LA VALIDATION
# ============================================
print("\n" + "="*50)
print("ÉVALUATION SUR LA VALIDATION")
print("="*50)

model.load_weights('best_cats_dogs_model.h5')
val_loss, val_accuracy = model.evaluate(val_flow, verbose=0)
print(f"Précision sur la validation: {val_accuracy:.4f}")
print(f"Perte sur la validation: {val_loss:.4f}")

# Matrice de confusion
def evaluate_confusion_matrix(model, val_flow):
    predictions = model.predict(val_flow, steps=val_flow.samples // batch_size)
    predicted_classes = (predictions > 0.5).astype(int).flatten()
    true_classes = []
    val_flow.reset()
    for _ in range(val_flow.samples // batch_size):
        _, labels = next(val_flow)
        true_classes.extend(labels.astype(int))
    true_classes = np.array(true_classes[:len(predicted_classes)])
    cm = confusion_matrix(true_classes, predicted_classes)
    
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Chat', 'Chien'], yticklabels=['Chat', 'Chien'], ax=axes[0])
    axes[0].set_title('Matrice de Confusion')
    axes[0].set_xlabel('Prédictions')
    axes[0].set_ylabel('Vraies Valeurs')
    
    report = classification_report(true_classes, predicted_classes, 
                                  target_names=['Chat', 'Chien'], output_dict=True)
    metrics_df = pd.DataFrame(report).T.iloc[:2, :3]
    metrics_df.plot(kind='bar', ax=axes[1])
    axes[1].set_title('Précision et Rappel par Classe')
    axes[1].set_xlabel('Classe')
    axes[1].set_ylabel('Score')
    axes[1].set_ylim([0, 1])
    axes[1].legend(loc='lower right')
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    print("\nRapport de classification:")
    print(classification_report(true_classes, predicted_classes, target_names=['Chat', 'Chien']))

evaluate_confusion_matrix(model, val_flow)

# ============================================
# PARTIE 9 : INFÉRENCE SUR LE TEST
# ============================================
print("\n" + "="*50)
print("INFÉRENCE SUR L'ENSEMBLE DE TEST")
print("="*50)

def generate_test_predictions(model, test_flow, threshold=0.5):
    predictions = model.predict(test_flow, steps=test_flow.samples // batch_size + 1)
    predictions = predictions.flatten()
    filenames = [os.path.basename(f) for f in test_flow.filenames]
    predictions = predictions[:len(filenames)]
    results_df = pd.DataFrame({
        'filename': filenames,
        'prob_dog': predictions,
        'predicted_label': ['dog' if p > threshold else 'cat' for p in predictions]
    })
    return results_df

test_predictions = generate_test_predictions(model, test_flow)
test_predictions.to_csv('test_predictions.csv', index=False)
print(f"✅ Prédictions générées: {len(test_predictions)} images")
print(test_predictions.head(10))

# ============================================
# PARTIE 10 : COMPARAISON BASE VS AUGMENTATION
# ============================================
print("\n" + "="*50)
print("COMPARAISON BASE VS AUGMENTATION")
print("="*50)

def train_baseline_model():
    base_train_gen = ImageDataGenerator(rescale=1./255)
    base_train_flow = base_train_gen.flow_from_dataframe(
        df_tr, x_col="filepath", y_col="label",
        target_size=(IMG_HEIGHT, IMG_WIDTH),
        class_mode="binary", batch_size=batch_size,
        shuffle=True, seed=seed, validate_filenames=False
    )
    base_model = create_cats_dogs_model()
    base_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                      loss='binary_crossentropy', metrics=['accuracy'])
    base_history = base_model.fit(
        base_train_flow,
        steps_per_epoch=base_train_flow.samples // batch_size,
        epochs=20,
        validation_data=val_flow,
        validation_steps=val_flow.samples // batch_size,
        callbacks=callbacks_list,
        verbose=0
    )
    return base_history, base_model

base_history, base_model = train_baseline_model()
base_val_acc = max(base_history.history['val_accuracy'])

print(f"Modèle avec augmentation: {val_accuracy:.4f}")
print(f"Modèle sans augmentation: {base_val_acc:.4f}")
print(f"Amélioration: +{(val_accuracy - base_val_acc)*100:.2f}%")

# ============================================
# PARTIE 11 : GESTION DES DÉSÉQUILIBRES
# ============================================
print("\n" + "="*50)
print("GESTION DES DÉSÉQUILIBRES")
print("="*50)

class_labels = df_tr['label'].values
class_weights = compute_class_weight('balanced', classes=np.unique(class_labels), y=class_labels)
class_weight_dict = dict(enumerate(class_weights))
print(f"Poids des classes: Chat={class_weight_dict[0]:.3f}, Chien={class_weight_dict[1]:.3f}")

# ============================================
# PARTIE 12 : SAUVEGARDE DES ARTEFACTS
# ============================================
print("\n" + "="*50)
print("SAUVEGARDE DES ARTEFACTS")
print("="*50)

model.save('cats_dogs_classifier.h5')
model.save('cats_dogs_saved_model', save_format='tf')

config = {
    'model': {'name': 'Cats vs Dogs Classifier', 'architecture': 'CNN with 5 blocks',
              'input_shape': [IMG_HEIGHT, IMG_WIDTH, 3], 'parameters': model.count_params(),
              'optimizer': 'Adam', 'learning_rate': 0.001, 'batch_size': batch_size},
    'data': {'train_samples': train_flow.samples, 'val_samples': val_flow.samples,
             'test_samples': test_flow.samples, 'classes': train_flow.class_indices},
    'augmentation': {'rotation_range': 45, 'width_shift_range': 0.15, 'height_shift_range': 0.15,
                     'zoom_range': 0.5, 'horizontal_flip': True},
    'performance': {'val_accuracy': float(val_accuracy), 'val_loss': float(val_loss)},
    'timestamp': datetime.now().isoformat()
}

with open('config.json', 'w') as f:
    json.dump(config, f, indent=2)
with open('config.yaml', 'w') as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Modèle sauvegardé: cats_dogs_classifier.h5")
print("✅ Configuration sauvegardée: config.json et config.yaml")
print("✅ Prédictions sauvegardées: test_predictions.csv")

# ============================================
# PARTIE 13 : EXTENSIONS (TRANSFER LEARNING)
# ============================================
print("\n" + "="*50)
print("EXTENSION : TRANSFER LEARNING AVEC MOBILENETV2")
print("="*50)

def create_transfer_learning_model(input_shape=(180, 180, 3)):
    base_model = tf.keras.applications.MobileNetV2(input_shape=input_shape, include_top=False, weights='imagenet')
    base_model.trainable = False
    model = tf.keras.Sequential([
        base_model,
        layers.GlobalAveragePooling2D(),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(1, activation='sigmoid')
    ])
    return model

transfer_model = create_transfer_learning_model()
transfer_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                      loss='binary_crossentropy', metrics=['accuracy'])
print("Modèle de transfer learning créé avec succès!")

# ============================================
# PARTIE 14 : RÉSUMÉ FINAL
# ============================================
print("\n" + "="*50)
print("RÉSUMÉ FINAL - DÉFI QUOTIDIEN")
print("="*50)
print(f"""
✅ Classification Chats vs Chiens réussie!

Performances:
- Précision sur la validation: {val_accuracy:.4f} ({val_accuracy*100:.2f}%)
- Perte sur la validation: {val_loss:.4f}
- Amélioration avec augmentation: +{(val_accuracy - base_val_acc)*100:.2f}%

Architecture:
- 5 blocs de convolution avec BatchNormalization
- Dropout: 0.3 (convolutionnel) et 0.5 (dense)
- GlobalAveragePooling2D pour la réduction
- 1 couche de sortie sigmoïde

Livrables:
1. best_cats_dogs_model.h5 - Meilleur modèle
2. cats_dogs_classifier.h5 - Modèle entraîné
3. test_predictions.csv - Prédictions sur le test
4. config.json et config.yaml - Configuration
5. Visualisations et métriques

Compétences acquises:
✅ Prétraitement d'images pour CNN
✅ Augmentation de données
✅ Architecture CNN pour classification binaire
✅ Dropout pour régularisation
✅ Évaluation et visualisation des performances
""")

print("="*50)
print("🎉 DÉFI QUOTIDIEN COMPLET! FÉLICITATIONS!")
print("="*50)

ModuleNotFoundError: No module named 'yaml'